# 🚀 02. Train Vision-Only Student Model (Phase 2 - CMKD)

> **Project:** Durian Ripeness Classification via Cross-Modal Knowledge Distillation (CMKD)  
> **Lead:** Leader / Member 1 (Vision Lead & Multi-Modal Framework)  
> **Reference:** `specification/project_specification.pdf` (Section 1.3.3) & `tmp_student.md`  
> **Notebook Role:**
1. Mount Google Drive & verify GPU availability.
2. `git clone` / `git pull` the latest source code from GitHub.
3. Link dataset (`data/final`), Teacher checkpoint (`checkpoints/teachermodal/best_teacher.pth`), and create target folders (`checkpoints/studentmodal`, `results_student`) on Google Drive.
4. Launch 2-phase distillation training (`train_student.py`) and export the benchmark comparison table.

In [13]:
# ==============================================================================
# 1. MOUNT GOOGLE DRIVE & CHECK GPU
# ==============================================================================
import os
import sys

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
if IN_COLAB:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    print("✅ Google Drive mounted successfully!")
else:
    print("ℹ️ Running in Local environment.")

!nvidia-smi

✅ Google Drive mounted successfully!
Fri Nov 20 20:33:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P0             47W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+----------

In [14]:
# ==============================================================================
# 2. SYNC LATEST CODE FROM GITHUB
# ==============================================================================
REPO_URL = "https://github.com/Cocobaut/durian-ripeness-classification.git"
BRANCH = "cocobaut"
REPO_DIR = "/content/durian-ripeness-classification"

if IN_COLAB:
    %cd /content
    if not os.path.exists(REPO_DIR):
        if BRANCH:
            !git clone -b $BRANCH $REPO_URL $REPO_DIR
        else:
            !git clone $REPO_URL $REPO_DIR
        %cd $REPO_DIR
    else:
        %cd $REPO_DIR
        # Remove temporary repo symlinks (without touching Drive files) to prevent git pull conflicts
        !find . -maxdepth 5 -type l -delete
        !git fetch origin
        if BRANCH:
            !git checkout -f $BRANCH
            !git reset --hard origin/$BRANCH
        else:
            !git reset --hard origin/$(git rev-parse --abbrev-ref HEAD)
else:
    cur = os.path.abspath(".")
    while not (os.path.isdir(os.path.join(cur, "src")) and os.path.isdir(os.path.join(cur, "workspaces"))):
        parent = os.path.dirname(cur)
        if parent == cur:
            break
        cur = parent
    os.chdir(cur)

print(f"✅ Working in project root directory: {os.getcwd()}")

/content
/content/durian-ripeness-classification
remote: Enumerating objects: 41, done.
remote: Counting objects: 100% (41/41), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 41 (delta 16), reused 41 (delta 16), pack-reused 0 (from 0)
Unpacking objects: 100% (41/41), 467.82 KiB | 12.64 MiB/s, done.
From https://github.com/Cocobaut/durian-ripeness-classification
   60b3fe0..194f4f5  cocobaut   -> origin/cocobaut
Already on 'cocobaut'
Your branch is behind 'origin/cocobaut' by 1 commit, and can be fast-forwarded.
  (use "git pull" to update your local branch)
HEAD is now at 194f4f5 [FIX] branch student
✅ Working in project root directory: /content/durian-ripeness-classification


In [15]:
# ==============================================================================
# 3. FAST DEPENDENCY INSTALLATION WITH `uv`
# ==============================================================================
!pip install -q uv
!uv pip install --system -r requirements.txt
print("✅ Installed all dependencies successfully!")

Using Python 3.13.16 environment at: /usr
Checked 22 packages in 103ms
✅ Installed all dependencies successfully!


In [16]:
# ==============================================================================
# 4. CONFIGURE CHECKPOINT & DATASET LINKS FROM GOOGLE DRIVE
# ==============================================================================
from pathlib import Path

if IN_COLAB:
    DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/durian-ripeness-cmkd")
    DRIVE_CHECKPOINTS = DRIVE_PROJECT_ROOT / "checkpoints"
    DRIVE_DATA = DRIVE_PROJECT_ROOT / "data" / "final"
    DRIVE_RESULTS_TEACHER = DRIVE_PROJECT_ROOT / "results_teacher"
    DRIVE_RESULTS_STUDENT = DRIVE_PROJECT_ROOT / "results_student"

    # Pre-create studentmodal & results_student folders on Google Drive for real-time persistence
    (DRIVE_CHECKPOINTS / "teachermodal").mkdir(parents=True, exist_ok=True)
    (DRIVE_CHECKPOINTS / "studentmodal").mkdir(parents=True, exist_ok=True)
    DRIVE_RESULTS_STUDENT.mkdir(parents=True, exist_ok=True)

    # Symlink repo directories to Google Drive so outputs persist across Colab disconnects
    !rm -rf checkpoints/unimodal checkpoints/teachermodal checkpoints/studentmodal
    !rm -rf workspaces/member1_vision_lead/04_train_teacher_and_student/results_student
    !mkdir -p checkpoints data/final
    !ln -sfn "{str(DRIVE_CHECKPOINTS / 'unimodal')}" checkpoints/unimodal
    !ln -sfn "{str(DRIVE_CHECKPOINTS / 'teachermodal')}" checkpoints/teachermodal
    !ln -sfn "{str(DRIVE_CHECKPOINTS / 'studentmodal')}" checkpoints/studentmodal
    !ln -sfn "{str(DRIVE_RESULTS_STUDENT)}" workspaces/member1_vision_lead/04_train_teacher_and_student/results_student
    if DRIVE_RESULTS_TEACHER.exists():
        !rm -rf workspaces/member1_vision_lead/04_train_teacher_and_student/results
        !ln -sfn "{str(DRIVE_RESULTS_TEACHER)}" workspaces/member1_vision_lead/04_train_teacher_and_student/results
    for split_name in ["train", "valid", "test"]:
        src_split = DRIVE_DATA / split_name
        if src_split.exists():
            !rm -rf data/final/{split_name}
            !ln -sfn "{str(src_split)}" data/final/{split_name}

Path("checkpoints/studentmodal").mkdir(parents=True, exist_ok=True)
print("✅ Verifying configured paths:")
print(" - Teacher ckpt   :", Path("checkpoints/teachermodal/best_teacher.pth").exists(), "->", Path("checkpoints/teachermodal").resolve())
print(" - Studentmodal   :", Path("checkpoints/studentmodal").resolve())
print(" - Results student:", Path("workspaces/member1_vision_lead/04_train_teacher_and_student/results_student").resolve())
print(" - Manifest CSV   :", Path("data/final/final_multimodal_manifest.csv").exists())

✅ Verifying configured paths:
 - Teacher ckpt   : True -> /content/drive/MyDrive/durian-ripeness-cmkd/checkpoints/teachermodal
 - Studentmodal   : /content/drive/MyDrive/durian-ripeness-cmkd/checkpoints/studentmodal
 - Results student: /content/drive/MyDrive/durian-ripeness-cmkd/results_student
 - Manifest CSV   : True


In [ ]:
# ==============================================================================
# 5. TRAIN 3 VISION-ONLY STUDENTS (SEEDS) VIA CMKD ON TRAIN + VAL (PHASE 2 - CMKD v3)
# ==============================================================================
# CMKD v3 = v2 recipe + two changes fixed BEFORE looking at Test:
#  1. Distill on Train + Val fruits (+28 fruits). The Teacher was trained on Train only, so its soft labels /
#     vision features on Val fruits are genuinely out-of-sample "dark knowledge".
#     -> Val is no longer held out: fixed-epoch training (--select-metric last, EMA weights at epoch 30).
#  2. 3 seeds -> report per-seed Test (mean ± std) AND the probability-averaged ensemble (cell 6).
# Teacher / unimodal checkpoints are reused (not retrained). ~30 min per seed on A100.
SEEDS = [42, 43, 44]
for seed in SEEDS:
    print(f"\n{'=' * 30} SEED {seed} {'=' * 30}")
    !python workspaces/member1_vision_lead/04_train_teacher_and_student/train_student.py \
        --manifest data/final/final_multimodal_manifest.csv \
        --data-dir data/final \
        --teacher-ckpt checkpoints/teachermodal/best_teacher.pth \
        --student-backbone convnextv2_base \
        --save-path checkpoints/studentmodal/student_trainval_s{seed}.pth \
        --results-dir workspaces/member1_vision_lead/04_train_teacher_and_student/results_student/seed_{seed} \
        --train-splits train,valid \
        --alpha 1.0 --beta 1.0 --gamma 0.3 --temperature 4.0 \
        --feat-target vision --feat-loss-type cosine_relational --kd-weighting teacher_correct \
        --loss-type cost_sensitive_focal --focal-gamma 2.0 --cost-weight 0.5 \
        --batch-size 4 --accum-steps 2 \
        --epochs 30 --warmup-epochs 5 \
        --lr 5e-4 --lr-backbone 5e-6 --min-lr 1e-6 \
        --ema-decay 0.99 \
        --select-metric last \
        --eval-test \
        --fresh \
        --seed {seed} \
        --device cuda

In [ ]:
# ==============================================================================
# 6. SEED ENSEMBLE EVALUATION ON TEST + PER-SEED SUMMARY + BENCHMARK TABLE
# ==============================================================================
import json
import pandas as pd

RES = Path("workspaces/member1_vision_lead/04_train_teacher_and_student/results_student")
ckpts = " ".join(f"checkpoints/studentmodal/student_trainval_s{s}.pth" for s in SEEDS)

# Probability-averaged ensemble of the 3 Students (same 3-view TTA as the unimodal baseline).
# Writes test_* files + benchmark_comparison.md into results_student/.
!python workspaces/member1_vision_lead/04_train_teacher_and_student/eval_student.py \
    --checkpoint {ckpts} \
    --manifest data/final/final_multimodal_manifest.csv \
    --data-dir data/final \
    --split test \
    --output-dir {str(RES)}

# Per-seed Test results (single models) -> mean ± std, to show the ensemble gain is not a lucky seed
rows = []
for s in SEEDS:
    m = json.loads((RES / f"seed_{s}" / "test_metrics.json").read_text(encoding="utf-8"))["metrics"]
    rows.append({"seed": s, "fruit_acc": m["fruit_accuracy"], "fruit_macro_f1": m["fruit_macro_f1"]})
ens = json.loads((RES / "test_metrics.json").read_text(encoding="utf-8"))["metrics"]
per_seed = pd.DataFrame(rows)
summary = pd.concat([
    per_seed,
    pd.DataFrame([{"seed": "mean ± std",
                   "fruit_acc": f"{per_seed.fruit_acc.mean():.4f} ± {per_seed.fruit_acc.std():.4f}",
                   "fruit_macro_f1": f"{per_seed.fruit_macro_f1.mean():.4f} ± {per_seed.fruit_macro_f1.std():.4f}"},
                  {"seed": "ENSEMBLE", "fruit_acc": ens["fruit_accuracy"], "fruit_macro_f1": ens["fruit_macro_f1"]}]),
], ignore_index=True)
summary.to_csv(RES / "seed_summary.csv", index=False)
print(summary.to_string(index=False))

comp_md = RES / "benchmark_comparison.md"
if comp_md.is_file():
    print("\n" + comp_md.read_text(encoding="utf-8"))
print("✅ All Student evaluation results saved at:", RES.resolve())